# Step 6 - Model Evaluation (V2)

## Purpose

This notebook replaces the label handling in the original Step 6 evaluation. The CNN in Step 5 was trained with **folder-derived target labels**, because the dataset's `Disease_Type` metadata disagrees systematically with the Healthy and Leaf Blight folders. This evaluation therefore derives `Target_Class` from each resolved image's parent folder and uses that same definition for the test labels.

The original metadata label is retained only for traceability and comparison. It is **not** used as the ground truth for the V2 metrics. The filename is used to locate the file, but only decoded RGB pixels are passed to the CNN.

## Where to run this notebook

Run this notebook in the environment that contains all three required inputs:

1. this Git repository;
2. the complete GVLiD image folders; and
3. `cnn_final.keras` produced by Step 5.

Google Colab is recommended if the model and dataset are already stored in Google Drive. Mount Drive, clone or open the latest repository, and edit only the two optional path overrides in the configuration cell below. Evaluation does not require model retraining and can also run locally if TensorFlow, the images and the saved model are available.

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

## Configuration

The automatic project-root discovery works when the notebook is run from the repository root or its `notebooks` directory. In Colab, set `RAW_DATA_OVERRIDE` and `MODEL_PATH_OVERRIDE` if the large files are stored in Google Drive.

In [ ]:
# Optional Colab setup (uncomment if needed):
# from google.colab import drive
# drive.mount('/content/drive')

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'processed' / 'dataset_split.csv').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the project root containing data/processed/dataset_split.csv.'
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src import RANDOM_SEED

# Set either value to a Path if Colab/Drive stores the files elsewhere.
RAW_DATA_OVERRIDE = None
MODEL_PATH_OVERRIDE = None
# Example:
# RAW_DATA_OVERRIDE = Path('/content/drive/MyDrive/grapevine/GVLiD')
# MODEL_PATH_OVERRIDE = Path('/content/drive/MyDrive/grapevine/cnn_final.keras')

default_raw_root = PROJECT_ROOT / 'data' / 'raw' / 'GVLiD'
if not default_raw_root.exists():
    default_raw_root = PROJECT_ROOT / 'data' / 'raw'

RAW_DATA_DIR = Path(RAW_DATA_OVERRIDE) if RAW_DATA_OVERRIDE else default_raw_root
MODEL_PATH = (
    Path(MODEL_PATH_OVERRIDE)
    if MODEL_PATH_OVERRIDE
    else PROJECT_ROOT / 'results' / 'cnn_final.keras'
)
SPLIT_FILE = PROJECT_ROOT / 'data' / 'processed' / 'dataset_split.csv'
HISTORY_PATH = PROJECT_ROOT / 'results' / 'training_history.pkl'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'evaluation_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['Black Rot', 'Esca', 'Healthy', 'Leaf Blight']
LABEL_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print('Project root:', PROJECT_ROOT)
print('Raw data:', RAW_DATA_DIR)
print('Model:', MODEL_PATH)
print('Split file:', SPLIT_FILE)

In [ ]:
required_paths = {
    'raw image directory': RAW_DATA_DIR,
    'trained model': MODEL_PATH,
    'split manifest': SPLIT_FILE,
}
missing_inputs = {name: path for name, path in required_paths.items() if not path.exists()}
if missing_inputs:
    details = '\n'.join(f'- {name}: {path}' for name, path in missing_inputs.items())
    raise FileNotFoundError(
        'Required evaluation inputs are missing. Correct the configuration paths before continuing:\n'
        + details
    )

print('All required inputs are available.')

## Resolve images and reconstruct the modelling target

This is the key V2 correction. Image files are located recursively, and `Target_Class` is derived from the resolved image's class folder. `Disease_Type` remains in the table only so the disagreement can be audited.

In [ ]:
split_df = pd.read_csv(SPLIT_FILE)
required_columns = {'Image_ID', 'image_filename', 'Vineyard', 'Disease_Type', 'Split'}
missing_columns = required_columns - set(split_df.columns)
if missing_columns:
    raise ValueError(f'Split manifest is missing columns: {sorted(missing_columns)}')

folder_to_class = {
    'black rot': 'Black Rot',
    'esca': 'Esca',
    'healthy': 'Healthy',
    'leaf blight': 'Leaf Blight',
}
image_extensions = {'.jpg', '.jpeg', '.png'}
paths_by_filename = defaultdict(list)

for path in RAW_DATA_DIR.rglob('*'):
    if path.is_file() and path.suffix.lower() in image_extensions:
        parent_key = path.parent.name.strip().lower().replace('_', ' ')
        if parent_key in folder_to_class:
            paths_by_filename[path.name.lower()].append(path)

duplicate_filename_count = sum(
    len(paths) > 1 for paths in paths_by_filename.values()
)
if duplicate_filename_count:
    print(
        f'Note: {duplicate_filename_count} filenames occur in more than one dataset copy. '
        'V2 verifies that duplicate candidates belong to the same class and selects '
        'the first path deterministically.'
    )

def resolve_image(filename: str):
    candidates = sorted(
        paths_by_filename.get(str(filename).lower(), []),
        key=lambda path: str(path).lower(),
    )
    if not candidates:
        return None
    candidate_classes = {
        folder_to_class[p.parent.name.strip().lower().replace('_', ' ')]
        for p in candidates
    }
    if len(candidate_classes) != 1:
        raise ValueError(
            f'Filename {filename!r} appears under conflicting class folders: '
            f'{sorted(candidate_classes)}'
        )
    return candidates[0]

split_df['image_path'] = split_df['image_filename'].map(resolve_image)
missing_images = split_df[split_df['image_path'].isna()]
if not missing_images.empty:
    raise FileNotFoundError(
        f'{len(missing_images)} manifest images could not be found beneath {RAW_DATA_DIR}. '
        f'Examples: {missing_images.image_filename.head().tolist()}'
    )

split_df['Target_Class'] = split_df['image_path'].map(
    lambda path: folder_to_class[path.parent.name.strip().lower().replace('_', ' ')]
)
split_df['label'] = split_df['Target_Class'].map(LABEL_TO_INDEX).astype(np.int32)
split_df['metadata_agrees_with_target'] = (
    split_df['Disease_Type'] == split_df['Target_Class']
)

print('Manifest rows:', len(split_df))
print('Resolved images:', split_df['image_path'].notna().sum())
print('Metadata/folder disagreements:', (~split_df['metadata_agrees_with_target']).sum())
display(pd.crosstab(split_df['Target_Class'], split_df['Disease_Type']))

In [ ]:
test_df = split_df[split_df['Split'].str.lower() == 'test'].copy().reset_index(drop=True)

if test_df.empty:
    raise ValueError('The split manifest contains no Test rows.')
if test_df['Image_ID'].duplicated().any():
    raise ValueError('Duplicate Image_ID values were found in the test subset.')
if set(test_df['Target_Class']) != set(CLASS_NAMES):
    raise ValueError(
        'The test subset does not contain exactly the four expected target classes: '
        f'{sorted(test_df.Target_Class.unique())}'
    )

print('Test images:', len(test_df))
print('Corrected folder-derived test distribution:')
display(test_df['Target_Class'].value_counts().reindex(CLASS_NAMES).rename('support'))
print('Metadata labels are shown below for comparison only:')
display(test_df['Disease_Type'].value_counts().reindex(CLASS_NAMES).rename('metadata_support'))

## Build the test pipeline

Only decoded, resized and normalised RGB pixels are passed to the CNN. Filenames, folder names and metadata are not model inputs. The test dataset is not shuffled so predictions remain aligned with `test_df`.

In [ ]:
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

paths = test_df['image_path'].astype(str).to_numpy()
labels = test_df['label'].to_numpy(dtype=np.int32)
test_ds = tf.data.Dataset.from_tensor_slices((paths, labels))
test_ds = test_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print('Test batches:', int(tf.data.experimental.cardinality(test_ds)))

## Load and validate the trained model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
input_shape = tuple(model.input_shape[1:])
expected_input_shape = (*IMG_SIZE, 3)
if input_shape != expected_input_shape:
    raise ValueError(
        f'Model input shape is {input_shape}; expected {expected_input_shape} from Step 5.'
    )
output_units = int(model.output_shape[-1])
if output_units != len(CLASS_NAMES):
    raise ValueError(
        f'Model has {output_units} output units; expected {len(CLASS_NAMES)} for {CLASS_NAMES}.'
    )

print('Model loaded successfully.')
print('Model input shape:', input_shape)
print('Model output classes:', output_units)

## Generate predictions using corrected test labels

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
y_prob = model.predict(test_ds, verbose=1)
y_true = labels
y_pred = np.argmax(y_prob, axis=1)

if len(y_pred) != len(test_df):
    raise RuntimeError('Prediction count does not match the test manifest.')
prediction_accuracy = accuracy_score(y_true, y_pred)
if not np.isclose(test_accuracy, prediction_accuracy, atol=1e-6):
    raise RuntimeError(
        f'model.evaluate accuracy ({test_accuracy}) does not match prediction accuracy '
        f'({prediction_accuracy}). Check test ordering and preprocessing.'
    )

print(f'Corrected test loss: {test_loss:.4f}')
print(f'Corrected test accuracy: {test_accuracy:.4f}')

## Multiclass evaluation

Macro metrics give every class equal importance. The majority-class baseline is included so the CNN is not judged by accuracy in isolation.

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
majority_baseline = test_df['Target_Class'].value_counts(normalize=True).max()

summary_df = pd.DataFrame({
    'Metric': [
        'Test loss', 'Test accuracy', 'Macro precision', 'Macro recall',
        'Macro F1', 'Weighted F1', 'Majority-class accuracy baseline'
    ],
    'Score': [
        test_loss, accuracy, macro_precision, macro_recall,
        macro_f1, weighted_f1, majority_baseline
    ],
})
display(summary_df.round(4))

report_dict = classification_report(
    y_true,
    y_pred,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df.round(4))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
cm_normalized = confusion_matrix(
    y_true, y_pred, labels=range(len(CLASS_NAMES)), normalize='true'
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0]
)
axes[0].set(title='Confusion Matrix - Counts', xlabel='Predicted', ylabel='Actual')
sns.heatmap(
    cm_normalized, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1]
)
axes[1].set(title='Confusion Matrix - Row Normalised', xlabel='Predicted', ylabel='Actual')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices_v2.png', dpi=300, bbox_inches='tight')
plt.show()

## Decision-support view: disease triage

The four-class metrics evaluate disease classification. For the proposed inspection-prioritisation use case, it is also useful to ask whether the model separates **any disease** from Healthy. Disease recall measures how many truly diseased test images would be flagged for review. This binary view supplements rather than replaces the four-class evaluation.

In [ ]:
healthy_index = LABEL_TO_INDEX['Healthy']
y_true_disease = (y_true != healthy_index).astype(int)
y_pred_disease = (y_pred != healthy_index).astype(int)

triage_precision = precision_score(y_true_disease, y_pred_disease, zero_division=0)
triage_recall = recall_score(y_true_disease, y_pred_disease, zero_division=0)
triage_f1 = f1_score(y_true_disease, y_pred_disease, zero_division=0)
triage_cm = confusion_matrix(y_true_disease, y_pred_disease, labels=[0, 1])
review_rate = y_pred_disease.mean()

triage_df = pd.DataFrame({
    'Metric': ['Disease precision', 'Disease recall', 'Disease F1', 'Proportion flagged for review'],
    'Score': [triage_precision, triage_recall, triage_f1, review_rate],
})
display(triage_df.round(4))

sns.heatmap(
    triage_cm, annot=True, fmt='d', cmap='Greens',
    xticklabels=['Healthy', 'Disease'], yticklabels=['Healthy', 'Disease']
)
plt.title('Inspection-triage Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'triage_confusion_matrix_v2.png', dpi=300, bbox_inches='tight')
plt.show()

## Error analysis

High-confidence errors are especially important because softmax confidence is not proof of correctness. Review these examples for symptom similarity, background cues, image quality and possible remaining label problems.

In [ ]:
predictions_df = test_df[[
    'Image_ID', 'image_filename', 'Vineyard', 'Disease_Type',
    'Target_Class', 'metadata_agrees_with_target', 'image_path'
]].copy()
predictions_df['Predicted_Class'] = [CLASS_NAMES[i] for i in y_pred]
predictions_df['Confidence'] = y_prob.max(axis=1)
predictions_df['Correct'] = (y_true == y_pred)
for index, class_name in enumerate(CLASS_NAMES):
    predictions_df[f'Probability_{class_name}'] = y_prob[:, index]

errors_df = predictions_df[~predictions_df['Correct']].sort_values(
    'Confidence', ascending=False
)
print('Incorrect predictions:', len(errors_df))
display(errors_df.head(10))

sample_errors = errors_df.head(12)
if not sample_errors.empty:
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    for axis in axes.flat:
        axis.axis('off')
    for axis, (_, row) in zip(axes.flat, sample_errors.iterrows()):
        image = tf.keras.utils.load_img(row['image_path'], target_size=IMG_SIZE)
        axis.imshow(image)
        axis.set_title(
            f"Actual: {row['Target_Class']}\n"
            f"Pred: {row['Predicted_Class']} ({row['Confidence']:.2f})"
        )
        axis.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'high_confidence_errors_v2.png', dpi=300, bbox_inches='tight')
    plt.show()

## Training history (if available)

Training and validation curves describe model fitting. They must not be substituted for the corrected held-out test metrics above.

In [ ]:
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, 'rb') as file:
        history = pickle.load(file)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history['accuracy'], label='Training')
    axes[0].plot(history['val_accuracy'], label='Validation')
    axes[0].set(title='Training and Validation Accuracy', xlabel='Epoch', ylabel='Accuracy')
    axes[0].legend()
    axes[1].plot(history['loss'], label='Training')
    axes[1].plot(history['val_loss'], label='Validation')
    axes[1].set(title='Training and Validation Loss', xlabel='Epoch', ylabel='Loss')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_history_v2.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Training history not found; skipping curves:', HISTORY_PATH)

## Save corrected results

V2 outputs use separate filenames and do not overwrite the original, invalid evaluation results.

In [ ]:
summary_df.to_csv(OUTPUT_DIR / 'summary_metrics_v2.csv', index=False)
report_df.to_csv(OUTPUT_DIR / 'classification_report_v2.csv')
triage_df.to_csv(OUTPUT_DIR / 'triage_metrics_v2.csv', index=False)
predictions_df.drop(columns=['image_path']).to_csv(
    OUTPUT_DIR / 'test_predictions_v2.csv', index=False
)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    OUTPUT_DIR / 'confusion_matrix_counts_v2.csv'
)

print('Corrected evaluation outputs saved to:', OUTPUT_DIR)

## Interpretation checklist

Complete the final narrative only after running all cells and reviewing the corrected outputs:

- Compare corrected test accuracy with the majority-class baseline.
- Use macro F1 and per-class recall to judge whether performance is balanced.
- Identify the most important confusion pairs.
- Report disease-triage recall and the proportion of images that would be referred for human review.
- Inspect high-confidence errors and acknowledge that softmax scores are not calibrated certainty.
- State that the folder-derived target is a provisional label decision caused by a systematic metadata conflict.
- Do not claim automated diagnosis or treatment. The proposed role is prioritising human vineyard inspection.
- Make a transparent No-Go, continue-pilot, or limited-pilot recommendation based on the corrected evidence.